# 6 - Maintenance et réordonnancement

DuckLake n'a pas d'index : l'élagage repose sur les statistiques min/max **par fichier** (`ducklake_file_column_stats`, utilisées par le planificateur — « Total Files Read » dans `EXPLAIN ANALYZE`) et par row group Parquet. Elles ne servent que si les données sont physiquement groupées selon `dataset_metadata.cluster_by`. Chaque update écrit de nouveaux fichiers triés dans le lot, mais pas dans l'ordre global : les plages des fichiers se recouvrent peu à peu.

Ce notebook montre :
1. la mesure de l'état du stockage (`storage_report`) et du recouvrement ;
2. l'effet de `recluster` sur le nombre de fichiers lus par une requête filtrée ;
3. la maintenance conditionnelle `maintain(MaintenancePolicy(...))` : étapes sautées, recluster opt-in, `dry_run`.

In [ ]:
# Importation des modules
import re
import shutil
import warnings
from pathlib import Path

import polars as pl

from dt_ducklake_manager import (
    DuckLakeConnector,
    DuckLakeMaintenance,
    DuckLakeTablesBuilder,
    MaintenancePolicy,
)

# Répertoire de travail réinitialisé à chaque exécution
workdir = Path("../outputs/notebook_6")
shutil.rmtree(workdir, ignore_errors=True)
(workdir / "data").mkdir(parents=True)

# Connexion : pas d'inlining (tout va en Parquet) et petite taille cible de fichier
# pour obtenir plusieurs fichiers avec un volume modeste
conn = DuckLakeConnector(
    str(workdir / "catalog.ducklake"),
    str(workdir / "data"),
    data_inlining_row_limit=0,
    ducklake_options={"target_file_size": "200KB"},
).connect()

## 1. Construction triée sur `cluster_by`

`build_schema(cluster_by=["k"])` écrit la table des faits `ORDER BY k` et persiste la clé de tri dans `dataset_metadata`.

In [ ]:
N = 75_000
df = pl.DataFrame(
    {
        "id": list(range(N)),
        "k": [(i * 7919) % 1000 for i in range(N)],
        "v": [float(i) for i in range(N)],
    }
)
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    DuckLakeTablesBuilder(
        df, categorical_threshold=4, primary_keys=["id"], connection=conn
    ).build_schema(cluster_by=["k"])

maint = DuckLakeMaintenance(conn)
print(maint.storage_report().summary())

## 2. Dégradation par des écritures non triées

Trois lots couvrant chacun toute la plage de `k` : chaque nouveau fichier chevauche tous les autres.

In [ ]:
def insert_unsorted_batches(first_batch: int, n_batches: int) -> None:
    """Insert unsorted batches spanning the whole k range."""
    for batch in range(first_batch, first_batch + n_batches):
        conn.execute(
            f"INSERT INTO db.main.fact_table SELECT range + {batch * N},"
            f" (range * 7919 + {batch}) % 1000, range::DOUBLE FROM range({N})"
        )


def file_ranges() -> pl.DataFrame:
    """Per-file min/max of k for the active files (annexe A)."""
    return conn.execute(
        """
        SELECT f.data_file_id, f.record_count,
               CAST(s.min_value AS INTEGER) AS k_min,
               CAST(s.max_value AS INTEGER) AS k_max
        FROM __ducklake_metadata_db.ducklake_data_file f
        JOIN __ducklake_metadata_db.ducklake_file_column_stats s USING (data_file_id)
        JOIN __ducklake_metadata_db.ducklake_column c
            ON c.column_id = s.column_id AND c.table_id = f.table_id
        WHERE c.column_name = 'k' AND f.end_snapshot IS NULL AND c.end_snapshot IS NULL
        ORDER BY k_min, k_max
        """
    ).pl()


insert_unsorted_batches(1, 3)
print(maint.storage_report().summary())
file_ranges()

## 3. Fichiers lus par une requête filtrée

`EXPLAIN ANALYZE` affiche `Total Files Read` : avec des plages qui se recouvrent, un filtre `k = 5` ne permet d'écarter aucun fichier.

In [ ]:
def total_files_read(query: str) -> int:
    """Extract 'Total Files Read' from the EXPLAIN ANALYZE plan of a query."""
    plan = conn.execute(f"EXPLAIN ANALYZE {query}").fetchall()[0][1]
    return sum(int(n) for n in re.findall(r"Total Files Read:\s*(\d+)", plan))


filtered_query = "SELECT count(*) FROM db.main.fact_table WHERE k = 5"
files_read_before = total_files_read(filtered_query)
print(f"Total Files Read avant recluster : {files_read_before}")

## 4. `recluster`

Réécriture complète dans une transaction (`CREATE TEMP TABLE` → `DELETE` → `INSERT … ORDER BY` avec `threads = 1`, restauré ensuite), puis `merge_adjacent_files`. La table conserve son identité et son historique.

**Coût** : réécriture complète ; l'espace disque est doublé jusqu'à `expire_snapshots` + `cleanup_files` (maintenance planifiée, jamais lancés par `recluster`).

In [ ]:
report = maint.recluster()
print(report.summary())
print(report.maintenance)

files_read_after = total_files_read(filtered_query)
print(f"Total Files Read : {files_read_before} -> {files_read_after}")
file_ranges()

Les fichiers sont désormais disjoints et monotones ; deux fichiers voisins peuvent partager une valeur limite (clé dupliquée), ce qui ne compte pas comme un recouvrement.

## 5. Maintenance conditionnelle : `maintain`

`maintain` lit `storage_report()` et n'exécute chaque étape (flush → rewrite → merge → recluster → expire → cleanup → delete_orphaned) que si son indicateur la justifie ; chaque étape sautée est journalisée avec sa raison et marquée `<étape>_skipped` dans le rapport.

### 5.1 Politique par défaut sur une table saine
Rien n'est justifié : aucun fichier de suppression, recluster non activé, aucune rétention.

In [ ]:
report = maint.maintain(MaintenancePolicy(max_small_files=100))
{key: value for key, value in report.maintenance.items()}

### 5.2 Nouvelle dégradation, `dry_run` puis exécution
`recluster=True` est un opt-in ; `dry_run=True` journalise ce qui serait fait sans rien modifier.

In [ ]:
insert_unsorted_batches(4, 2)
conn.execute("UPDATE db.main.fact_table SET v = -1 WHERE id < 5000")
print(maint.storage_report().summary())

policy = MaintenancePolicy(recluster=True, max_overlap_ratio=0.3, max_small_files=100)

dry = maint.maintain(
    MaintenancePolicy(
        recluster=True, max_overlap_ratio=0.3, max_small_files=100, dry_run=True
    )
)
print("dry_run :", dry.maintenance)
print(f"Total Files Read (après dry_run) : {total_files_read(filtered_query)}")

In [ ]:
real = maint.maintain(policy)
print("exécution :", real.maintenance)
print(f"Total Files Read (après maintain) : {total_files_read(filtered_query)}")
print(maint.storage_report().summary())

### 5.3 Maintenance planifiée
`retention_days` rend l'expiration explicite (`None` = jamais) ; `full_maintenance(schema, table, older_than_days)` équivaut à `maintain(MaintenancePolicy(retention_days=older_than_days, max_small_files=1))`.

In [ ]:
planned = maint.maintain(MaintenancePolicy(retention_days=30, dry_run=True))
print(planned.maintenance)

conn.close()